# Clean OCR Validation Notebook

This notebook keeps only the required validation logic.

Main/default approach: **hybrid validation**.

- All existing fields except vendor are validated from the ground-truth `OCRed Text` using regex/parsing.
- Vendor/seller is validated from the ground-truth `Json Data` using `normalize_invoice()`.
- The final report uses the same reporting structure for all fields.

An optional all-JSON validation function is also included at the end, but the recommended default is the hybrid approach.


In [1]:
# ============================================================================
# CONFIGURATION
# ============================================================================

import json
import re
import warnings
from datetime import datetime
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")

# Fuzzy matching dependency.
# Uses fuzzywuzzy when available, otherwise falls back to Python's difflib.
try:
    from fuzzywuzzy import fuzz

    def fuzzy_score(a, b):
        return fuzz.token_set_ratio(str(a), str(b))

except ImportError:
    from difflib import SequenceMatcher

    def fuzzy_score(a, b):
        return int(SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio() * 100)


# ---------------------------------------------------------------------------
# Edit these paths before running.
# ---------------------------------------------------------------------------

GT_CSV_PATH = r"C:\Users\fkorniotis\code\makeathlon\makeathon-2026-WHOAMI-inform\.data\tesseract_validation\batch1_1.csv"
MODEL_PREDS_PATH = r"C:\Users\fkorniotis\code\makeathlon\makeathon-2026-WHOAMI-inform\.data\tesseract_validation\model preds\output_batch_30_random.csv"

# Validation parameters
FUZZY_MATCH_THRESHOLD = 90
STRICT_NUMERIC_TOLERANCE = 0.1

# Required ground-truth columns in batch1_1.csv
GT_FILENAME_COLUMN = "File Name"
GT_OCR_TEXT_COLUMN = "OCRed Text"
GT_JSON_COLUMN = "Json Data"

# Prediction filename column. The notebook will also auto-detect common alternatives.
PRED_FILENAME_COLUMN = "filename"

# Hybrid validation:
# Keep seller/vendor OUT of this list because vendor is validated from JSON separately.
HYBRID_NON_VENDOR_DOMAINS = [
    "Client Name",
    "Seller Tax ID",
    "Client Tax ID",
    "Invoice Number",
    "Invoice Date",
    "Net Worth",
    "VAT",
    "Gross Worth",
]

# Vendor/seller prediction column candidates in your model output.
# The first existing column will be used.
VENDOR_PREDICTION_COLUMN_CANDIDATES = [
    "vendor",
    "Seller Name",
    "seller",
    "seller_name",
    "vendor_name",
]


In [2]:
# ============================================================================
# DATA LOADING AND COLUMN NORMALIZATION
# ============================================================================

def detect_column(df, candidates, required=True, label="column"):
    """Return the first candidate column that exists in df."""
    for col in candidates:
        if col in df.columns:
            return col

    if required:
        raise KeyError(
            f"Could not find {label}. Tried: {candidates}. "
            f"Available columns: {df.columns.tolist()}"
        )

    return None


def filename_only(value):
    """Normalize paths to only the final filename."""
    if pd.isna(value):
        return value
    return str(value).replace("\\", "/").split("/")[-1]


def load_validation_data(gt_csv_path=GT_CSV_PATH, model_preds_path=MODEL_PREDS_PATH):
    """Load ground truth and model predictions, then normalize filename columns."""
    batch1_df = pd.read_csv(gt_csv_path)
    output_df = pd.read_csv(model_preds_path)

    pred_filename_col = detect_column(
        output_df,
        ["filename", "File Name", "Filename", "file_name", "image", "img_path"],
        required=True,
        label="prediction filename column",
    )

    if pred_filename_col != "filename":
        output_df = output_df.rename(columns={pred_filename_col: "filename"})

    if GT_FILENAME_COLUMN not in batch1_df.columns:
        raise KeyError(
            f"Ground-truth CSV must contain '{GT_FILENAME_COLUMN}'. "
            f"Available columns: {batch1_df.columns.tolist()}"
        )

    output_df["filename"] = output_df["filename"].map(filename_only)
    batch1_df[GT_FILENAME_COLUMN] = batch1_df[GT_FILENAME_COLUMN].map(filename_only)

    return batch1_df, output_df


batch1_df, output_df = load_validation_data()

print(f"Ground truth rows: {len(batch1_df)}")
print(f"Prediction rows:   {len(output_df)}")
print("\nGround-truth columns:")
print(batch1_df.columns.tolist())
print("\nPrediction columns:")
print(output_df.columns.tolist())


Ground truth rows: 499
Prediction rows:   30

Ground-truth columns:
['File Name', 'Json Data', 'OCRed Text']

Prediction columns:
['filename', 'doc_id', 'doc_type', 'vendor', 'vendor_normalized', 'invoice_number', 'invoice_date', 'due_date', 'currency', 'subtotal', 'tax', 'total', 'status', 'line_items', 'raw_text']


In [3]:
# ============================================================================
# REGEX / OCR-TEXT GROUND TRUTH PARSING HELPERS
# ============================================================================

def parse_ocred_text(ocred_text):
    """
    Extract the non-vendor ground-truth fields from OCRed Text.

    Vendor/seller is intentionally not used from this parser in the hybrid
    validation, because vendor is validated from Json Data.
    """
    extracted = {
        "Seller Name": None,
        "Client Name": None,
        "Seller Tax ID": None,
        "Client Tax ID": None,
        "Invoice Number": None,
        "Invoice Date": None,
        "Net Worth": None,
        "VAT": None,
        "Gross Worth": None,
    }

    if pd.isna(ocred_text):
        return extracted

    ocred_text = str(ocred_text)
    lines = ocred_text.split("\n")

    # Tax ID pattern: XXX-XX-XXXX
    tax_id_pattern = r"\d{3}-\d{2}-\d{4}"

    # Invoice number pattern
    invoice_num_pattern = r"Invoice\s+(?:no|number):\s*(\d+)"

    # Date patterns
    date_patterns = [
        r"Date\s+of\s+issue:\s*(\d{1,2}/\d{1,2}/\d{4})",
        r"Date\s+of\s+issue:\s*(\d{4}-\d{1,2}-\d{1,2})",
        r"Invoice\s+date:\s*(\d{1,2}/\d{1,2}/\d{4})",
        r"Invoice\s+date:\s*(\d{4}-\d{1,2}-\d{1,2})",
    ]

    # Invoice number
    for line in lines:
        match = re.search(invoice_num_pattern, line, re.IGNORECASE)
        if match:
            extracted["Invoice Number"] = match.group(1)
            break

    # Invoice date
    for line in lines:
        for pattern in date_patterns:
            match = re.search(pattern, line, re.IGNORECASE)
            if match:
                extracted["Invoice Date"] = match.group(1)
                break
        if extracted["Invoice Date"]:
            break

    # Tax IDs
    tax_ids = re.findall(tax_id_pattern, ocred_text)
    if len(tax_ids) >= 1:
        extracted["Seller Tax ID"] = tax_ids[0]
    if len(tax_ids) >= 2:
        extracted["Client Tax ID"] = tax_ids[1]

    # Names from OCRed Text
    seller_match = re.search(r"Seller:\s*([^\n]+)", ocred_text, re.IGNORECASE)
    if seller_match:
        extracted["Seller Name"] = seller_match.group(1).strip()

    client_match = re.search(r"Client:\s*([^\n]+)", ocred_text, re.IGNORECASE)
    if client_match:
        extracted["Client Name"] = client_match.group(1).strip()

    # Monetary values from SUMMARY section
    summary_match = re.search(
        r"SUMMARY.*?(?=Total|$)",
        ocred_text,
        re.IGNORECASE | re.DOTALL,
    )

    if summary_match:
        summary_text = summary_match.group(0)
        for line in summary_text.split("\n"):
            lower_line = line.lower()

            if "net worth" in lower_line:
                parts = re.findall(r"[\d\.]+", line)
                if parts:
                    extracted["Net Worth"] = float(parts[0])

            if "vat" in line.upper() and "gross" not in lower_line:
                parts = re.findall(r"[\d\.]+", line)
                if parts:
                    extracted["VAT"] = float(parts[0])

            if "gross worth" in lower_line:
                parts = re.findall(r"[\d\.]+", line)
                if parts:
                    extracted["Gross Worth"] = float(parts[0])

    return extracted


In [4]:
# ============================================================================
# JSON GROUND TRUTH HELPERS
# ============================================================================

def parse_json_value(value):
    """Safely parse a JSON cell that may already be a dict or may be a JSON string."""
    if value is None or pd.isna(value):
        return {}

    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        value = value.strip()
        if not value:
            return {}
        return json.loads(value)

    return value


def normalize_invoice(data):
    """
    Normalize the dataset's Json Data into flat invoice fields.

    The vendor field comes from:
        data["invoice"]["seller_name"]
    """
    data = data or {}
    invoice = data.get("invoice", {}) or {}
    items = data.get("items", []) or []
    subtotal_data = data.get("subtotal", {}) or {}
    payment = data.get("payment_instructions", {}) or {}

    def clean_string(value):
        if value is None:
            return None
        value = str(value).strip()
        return value if value else None

    def to_decimal(value):
        value = clean_string(value)
        if value is None:
            return None
        try:
            return Decimal(value.replace(",", ""))
        except (InvalidOperation, ValueError):
            return None

    def to_float(value):
        dec = to_decimal(value)
        return float(dec) if dec is not None else None

    def to_iso_date(value):
        value = clean_string(value)
        if not value:
            return None

        for fmt in ("%m/%d/%Y", "%Y-%m-%d", "%m-%d-%Y", "%d/%m/%Y", "%d-%m-%Y"):
            try:
                return datetime.strptime(value, fmt).date().isoformat()
            except ValueError:
                pass

        try:
            return datetime.fromisoformat(value).date().isoformat()
        except ValueError:
            return None

    def normalize_vendor_name(name):
        name = clean_string(name)
        if not name:
            return None
        return name.lower()

    vendor = clean_string(invoice.get("seller_name"))
    invoice_number = clean_string(invoice.get("invoice_number"))
    invoice_date = to_iso_date(invoice.get("invoice_date"))
    due_date = to_iso_date(invoice.get("due_date") or payment.get("due_date"))

    tax = to_float(subtotal_data.get("tax"))
    discount = to_float(subtotal_data.get("discount")) or 0.0
    total = to_float(subtotal_data.get("total"))

    computed_subtotal = None
    if total is not None:
        computed_subtotal = round(total - (tax or 0.0) + discount, 2)

    line_items = []
    for item in items:
        qty = to_float(item.get("quantity"))
        line_total = to_float(item.get("total_price"))
        unit_price = round(line_total / qty, 2) if qty and line_total is not None else None

        line_items.append({
            "description": clean_string(item.get("description")),
            "qty": qty,
            "unit_price": unit_price,
            "line_total": line_total,
        })

    item_descs = " | ".join(
        item.get("description", "").replace("\n", " ").strip()
        for item in items
        if item.get("description")
    )

    raw_text = (
        f"Invoice {invoice_number} from {vendor} dated {invoice_date}. "
        f"Items: {item_descs}. "
        f"Total: {total} USD."
    )

    return {
        "docid": f"inv{invoice_number}" if invoice_number else "inv_unknown",
        "doc_type": "invoice",
        "vendor": vendor,
        "vendor_normalized": normalize_vendor_name(vendor),
        "invoice_number": invoice_number,
        "invoice_date": invoice_date,
        "due_date": due_date,
        "currency": "USD",
        "subtotal": computed_subtotal,
        "tax": tax if tax is not None else 0.0,
        "total": total,
        "status": "open" if due_date else "unknown",
        "line_items": line_items,
        "raw_text": raw_text,
    }


def series_json_to_df(series):
    """Convert a pandas Series of dict-like values into a flat DataFrame."""
    return pd.json_normalize(series.map(lambda x: x if isinstance(x, dict) else {}), max_level=0)


def build_json_ground_truth_df(batch1_df):
    """Return a flat DataFrame created from the Json Data column."""
    if GT_JSON_COLUMN not in batch1_df.columns:
        raise KeyError(
            f"Ground-truth CSV must contain '{GT_JSON_COLUMN}'. "
            f"Available columns: {batch1_df.columns.tolist()}"
        )

    normalized_series = batch1_df[GT_JSON_COLUMN].map(lambda x: normalize_invoice(parse_json_value(x)))
    return series_json_to_df(normalized_series)


In [5]:
# ============================================================================
# COMPARISON HELPERS
# ============================================================================

def is_missing(value):
    """Return True for None/NaN/empty string."""
    if value is None:
        return True

    try:
        if pd.isna(value):
            return True
    except Exception:
        pass

    if isinstance(value, str) and value.strip() == "":
        return True

    return False


def extract_date_components(date_str):
    """
    Extract date components as (year, month, day).
    Handles YYYY-MM-DD, MM/DD/YYYY, DD/MM/YYYY, and dash variants.
    """
    if is_missing(date_str):
        return None

    date_str = str(date_str).strip()

    unambiguous_formats = [
        "%Y-%m-%d",
        "%Y/%m/%d",
    ]

    for date_format in unambiguous_formats:
        try:
            dt = datetime.strptime(date_str, date_format)
            return (dt.year, dt.month, dt.day)
        except ValueError:
            continue

    # Ambiguous forms: X/Y/YYYY or X-Y-YYYY.
    match = re.match(r"(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})", date_str)
    if match:
        first, second, year = map(int, match.groups())
        sep = "/" if "/" in date_str else "-"

        if first > 12:
            fmt = f"%d{sep}%m{sep}%Y"
            try:
                dt = datetime.strptime(date_str, fmt)
                return (dt.year, dt.month, dt.day)
            except ValueError:
                pass

        if second > 12:
            fmt = f"%m{sep}%d{sep}%Y"
            try:
                dt = datetime.strptime(date_str, fmt)
                return (dt.year, dt.month, dt.day)
            except ValueError:
                pass

        # If ambiguous, try US first, then EU.
        for fmt in (f"%m{sep}%d{sep}%Y", f"%d{sep}%m{sep}%Y"):
            try:
                dt = datetime.strptime(date_str, fmt)
                return (dt.year, dt.month, dt.day)
            except ValueError:
                continue

    return None


def fuzzy_compare(str1, str2, threshold=FUZZY_MATCH_THRESHOLD):
    """Compare two strings using fuzzy matching."""
    if is_missing(str1) or is_missing(str2):
        return 0, False

    str1 = str(str1).strip().lower()
    str2 = str(str2).strip().lower()

    if str1 == str2:
        return 100, True

    score = fuzzy_score(str1, str2)
    return score, score >= threshold


def strict_compare(val1, val2, tolerance=STRICT_NUMERIC_TOLERANCE):
    """Compare two numeric values with tolerance."""
    if is_missing(val1) or is_missing(val2):
        return False, None

    try:
        v1 = float(str(val1).replace(",", ""))
        v2 = float(str(val2).replace(",", ""))
        diff = abs(v1 - v2)
        return diff <= tolerance, diff
    except Exception:
        return False, None


def compare_values(domain, gt_value, pred_value, tolerance=STRICT_NUMERIC_TOLERANCE, fuzzy_threshold=FUZZY_MATCH_THRESHOLD):
    """
    Compare a single ground-truth value and prediction.
    Returns a detail dictionary.
    """
    numeric_domains = {
        "Net Worth",
        "VAT",
        "Gross Worth",
        "subtotal",
        "tax",
        "total",
        "invoice_number",
        "Invoice Number",
    }

    date_domains = {
        "Invoice Date",
        "invoice_date",
        "due_date",
    }

    if domain in date_domains:
        gt_components = extract_date_components(gt_value)
        pred_components = extract_date_components(pred_value)
        is_match = gt_components is not None and pred_components is not None and gt_components == pred_components

        return {
            "status": "match" if is_match else "mismatch",
            "ground_truth": gt_value,
            "prediction": pred_value,
            "ground_truth_components": gt_components,
            "prediction_components": pred_components,
            "score": 100 if is_match else 0,
        }

    if domain in numeric_domains:
        is_match, diff = strict_compare(gt_value, pred_value, tolerance=tolerance)
        return {
            "status": "match" if is_match else "mismatch",
            "ground_truth": gt_value,
            "prediction": pred_value,
            "difference": diff,
            "score": 100 if is_match else 0,
        }

    score, is_match = fuzzy_compare(str(gt_value), str(pred_value), threshold=fuzzy_threshold)
    return {
        "status": "match" if is_match else "mismatch",
        "ground_truth": gt_value,
        "prediction": pred_value,
        "similarity_score": score,
        "score": 100 if is_match else score,
    }


def initialize_results(domains, total_files):
    """Create the shared results structure used by all validators."""
    return {
        "domains_validated": domains,
        "total_files": total_files,
        "files_matched": 0,
        "files_missing": [],
        "domain_results": {
            domain: {
                "matches": 0,
                "mismatches": 0,
                "missing_in_ground_truth": 0,
                "missing_in_output": 0,
                "accuracy": 0.0,
                "details": [],
            }
            for domain in domains
        },
        "summary": {},
    }


def finalize_results(results):
    """Calculate domain-level and overall accuracy."""
    for domain in results["domains_validated"]:
        dr = results["domain_results"][domain]
        total_compared = dr["matches"] + dr["mismatches"]
        dr["accuracy"] = (dr["matches"] / total_compared) * 100 if total_compared > 0 else 0.0

    total_matches = sum(r["matches"] for r in results["domain_results"].values())
    total_comparisons = sum(
        r["matches"] + r["mismatches"]
        for r in results["domain_results"].values()
    )

    results["summary"] = {
        "total_domains": len(results["domains_validated"]),
        "total_comparisons": total_comparisons,
        "total_matches": total_matches,
        "overall_accuracy": (total_matches / total_comparisons * 100) if total_comparisons > 0 else 0.0,
    }

    return results


In [6]:
# ============================================================================
# MAIN SOLUTION: HYBRID VALIDATION
# ============================================================================

def build_gt_index(batch1_df):
    """Map filename to row index in the ground-truth DataFrame."""
    return {
        filename_only(row[GT_FILENAME_COLUMN]): idx
        for idx, row in batch1_df.iterrows()
    }


def get_prediction_vendor_column(output_df):
    """Find which prediction column contains vendor/seller output."""
    return detect_column(
        output_df,
        VENDOR_PREDICTION_COLUMN_CANDIDATES,
        required=True,
        label="vendor prediction column",
    )


def validate_hybrid(
    output_df,
    batch1_df,
    non_vendor_domains=None,
    include_vendor=True,
    tolerance=STRICT_NUMERIC_TOLERANCE,
    fuzzy_threshold=FUZZY_MATCH_THRESHOLD,
):
    """
    Hybrid validation.

    Ground truth source:
    - non-vendor domains: OCRed Text, parsed with regex
    - vendor: Json Data, normalized with normalize_invoice()

    Prediction source:
    - non-vendor domains: same-named columns in output_df
    - vendor: first existing column from VENDOR_PREDICTION_COLUMN_CANDIDATES
    """
    if non_vendor_domains is None:
        non_vendor_domains = HYBRID_NON_VENDOR_DOMAINS

    # Keep only domains available in the model output.
    available_non_vendor_domains = [
        domain for domain in non_vendor_domains
        if domain in output_df.columns
    ]

    missing_prediction_columns = [
        domain for domain in non_vendor_domains
        if domain not in output_df.columns
    ]

    domains = available_non_vendor_domains.copy()

    vendor_pred_col = None
    if include_vendor:
        vendor_pred_col = get_prediction_vendor_column(output_df)
        domains.append("vendor")

    results = initialize_results(domains=domains, total_files=len(output_df))

    if missing_prediction_columns:
        print("Warning: these non-vendor prediction columns were not found and will be skipped:")
        print(missing_prediction_columns)

    gt_index = build_gt_index(batch1_df)
    gt_json_df = build_json_ground_truth_df(batch1_df)

    for _, output_row in output_df.iterrows():
        filename = filename_only(output_row["filename"])

        if filename not in gt_index:
            results["files_missing"].append(filename)
            continue

        results["files_matched"] += 1
        gt_idx = gt_index[filename]
        gt_row = batch1_df.iloc[gt_idx]

        parsed_gt = parse_ocred_text(gt_row[GT_OCR_TEXT_COLUMN])

        # Non-vendor fields from OCRed Text.
        for domain in available_non_vendor_domains:
            gt_value = parsed_gt.get(domain)
            pred_value = output_row.get(domain)
            dr = results["domain_results"][domain]

            if is_missing(gt_value):
                dr["missing_in_ground_truth"] += 1
                dr["details"].append({
                    "filename": filename,
                    "status": "missing_in_gt",
                    "ground_truth": gt_value,
                    "prediction": pred_value,
                })
                continue

            if is_missing(pred_value):
                dr["missing_in_output"] += 1
                dr["mismatches"] += 1
                dr["details"].append({
                    "filename": filename,
                    "status": "missing_in_output",
                    "ground_truth": gt_value,
                    "prediction": pred_value,
                })
                continue

            detail = compare_values(
                domain=domain,
                gt_value=gt_value,
                pred_value=pred_value,
                tolerance=tolerance,
                fuzzy_threshold=fuzzy_threshold,
            )
            detail["filename"] = filename
            dr["details"].append(detail)

            if detail["status"] == "match":
                dr["matches"] += 1
            else:
                dr["mismatches"] += 1

        # Vendor from JSON ground truth.
        if include_vendor:
            domain = "vendor"
            gt_vendor = gt_json_df.iloc[gt_idx].get("vendor")
            pred_vendor = output_row.get(vendor_pred_col)
            dr = results["domain_results"][domain]

            if is_missing(gt_vendor):
                dr["missing_in_ground_truth"] += 1
                dr["details"].append({
                    "filename": filename,
                    "status": "missing_in_gt",
                    "ground_truth": gt_vendor,
                    "prediction": pred_vendor,
                })
                continue

            if is_missing(pred_vendor):
                dr["missing_in_output"] += 1
                dr["mismatches"] += 1
                dr["details"].append({
                    "filename": filename,
                    "status": "missing_in_output",
                    "ground_truth": gt_vendor,
                    "prediction": pred_vendor,
                })
                continue

            detail = compare_values(
                domain=domain,
                gt_value=gt_vendor,
                pred_value=pred_vendor,
                tolerance=tolerance,
                fuzzy_threshold=fuzzy_threshold,
            )
            detail["filename"] = filename
            detail["prediction_column"] = vendor_pred_col
            dr["details"].append(detail)

            if detail["status"] == "match":
                dr["matches"] += 1
            else:
                dr["mismatches"] += 1

    return finalize_results(results)


In [7]:
# ============================================================================
# REPORTING FUNCTIONS
# ============================================================================

def print_validation_report(validation_results, show_mismatches=True, show_all_details=False, max_mismatches_per_domain=10):
    """Print a formatted validation report."""
    results = validation_results

    print("\n" + "=" * 80)
    print("OCR VALIDATION REPORT")
    print("=" * 80)

    print(f"\nFiles validated: {results['files_matched']} / {results['total_files']}")

    if results["files_missing"]:
        print(f"Files missing from ground truth: {len(results['files_missing'])}")

    print("\n" + "-" * 80)
    print("OVERALL ACCURACY")
    print("-" * 80)
    print(f"Total comparisons: {results['summary']['total_comparisons']}")
    print(f"Total matches:      {results['summary']['total_matches']}")
    print(f"Overall accuracy:   {results['summary']['overall_accuracy']:.2f}%")

    print("\n" + "-" * 80)
    print("DOMAIN-BY-DOMAIN ACCURACY")
    print("-" * 80)

    rows = []
    for domain in results["domains_validated"]:
        dr = results["domain_results"][domain]
        rows.append({
            "Domain": domain,
            "Matches": dr["matches"],
            "Mismatches": dr["mismatches"],
            "Missing GT": dr["missing_in_ground_truth"],
            "Missing Pred": dr["missing_in_output"],
            "Accuracy %": f"{dr['accuracy']:.2f}%",
        })

    summary_df = pd.DataFrame(rows)
    display(summary_df)

    if show_mismatches or show_all_details:
        print("\n" + "-" * 80)
        print("DETAILED RESULTS")
        print("-" * 80)

        for domain in results["domains_validated"]:
            dr = results["domain_results"][domain]
            print(f"\n### {domain} ###")

            if show_all_details:
                details_to_show = dr["details"]
            else:
                details_to_show = [
                    d for d in dr["details"]
                    if d["status"] in {"mismatch", "missing_in_output"}
                ]

            if not details_to_show:
                print("  ✓ All matches!")
                continue

            print(f"  Showing {min(len(details_to_show), max_mismatches_per_domain)} of {len(details_to_show)} issue(s):")

            for detail in details_to_show[:max_mismatches_per_domain]:
                print(f"    {detail['filename']}: {detail['status']}")
                print(f"      GT:         {detail.get('ground_truth')}")
                print(f"      Prediction: {detail.get('prediction')}")

                if "similarity_score" in detail:
                    print(f"      Similarity: {detail['similarity_score']:.0f}%")

                if "difference" in detail and detail["difference"] is not None:
                    print(f"      Difference: {detail['difference']}")

                if "prediction_column" in detail:
                    print(f"      Prediction column used: {detail['prediction_column']}")

    print("\n" + "=" * 80)


def get_mismatch_dataframe(validation_results, domain=None):
    """Return mismatches and missing predictions as a DataFrame."""
    rows = []

    domains_to_check = [domain] if domain else validation_results["domains_validated"]

    for dom in domains_to_check:
        for detail in validation_results["domain_results"][dom]["details"]:
            if detail["status"] in {"mismatch", "missing_in_output"}:
                row = {
                    "Domain": dom,
                    "Filename": detail["filename"],
                    "Ground Truth": detail.get("ground_truth"),
                    "Prediction": detail.get("prediction"),
                    "Status": detail["status"],
                }

                if "similarity_score" in detail:
                    row["Similarity %"] = detail["similarity_score"]

                if "difference" in detail:
                    row["Difference"] = detail["difference"]

                if "prediction_column" in detail:
                    row["Prediction Column"] = detail["prediction_column"]

                rows.append(row)

    return pd.DataFrame(rows)


def print_final_summary(validation_results):
    """Print a compact final summary after the detailed report."""
    print("\n" + "=" * 80)
    print("FINAL VALIDATION SUMMARY")
    print("=" * 80)

    print(f"\nFiles validated: {validation_results['files_matched']} / {validation_results['total_files']}")
    print(f"Overall accuracy: {validation_results['summary']['overall_accuracy']:.2f}%")
    print(
        f"Total matches: {validation_results['summary']['total_matches']} / "
        f"{validation_results['summary']['total_comparisons']}"
    )

    perfect_domains = [
        domain
        for domain in validation_results["domains_validated"]
        if validation_results["domain_results"][domain]["accuracy"] == 100.0
        and (
            validation_results["domain_results"][domain]["matches"]
            + validation_results["domain_results"][domain]["mismatches"]
        ) > 0
    ]

    if perfect_domains:
        print(f"\nPerfect domains ({len(perfect_domains)}):")
        for domain in perfect_domains:
            print(f"  - {domain}")

    print("\n" + "=" * 80)


In [8]:
# ============================================================================
# RUN HYBRID VALIDATION
# ============================================================================

hybrid_results = validate_hybrid(
    output_df=output_df,
    batch1_df=batch1_df,
    non_vendor_domains=HYBRID_NON_VENDOR_DOMAINS,
    include_vendor=True,
)

print_validation_report(
    hybrid_results,
    show_mismatches=True,
    show_all_details=False,
    max_mismatches_per_domain=10,
)

print_final_summary(hybrid_results)

hybrid_mismatches_df = get_mismatch_dataframe(hybrid_results)
display(hybrid_mismatches_df.head(30))


['Client Name', 'Seller Tax ID', 'Client Tax ID', 'Invoice Number', 'Invoice Date', 'Net Worth', 'VAT', 'Gross Worth']

OCR VALIDATION REPORT

Files validated: 30 / 30

--------------------------------------------------------------------------------
OVERALL ACCURACY
--------------------------------------------------------------------------------
Total comparisons: 30
Total matches:      30
Overall accuracy:   100.00%

--------------------------------------------------------------------------------
DOMAIN-BY-DOMAIN ACCURACY
--------------------------------------------------------------------------------


,Domain,Matches,Mismatches,Missing GT,Missing Pred,Accuracy %
0,vendor,30,0,0,0,100.00%



--------------------------------------------------------------------------------
DETAILED RESULTS
--------------------------------------------------------------------------------

### vendor ###
  ✓ All matches!


FINAL VALIDATION SUMMARY

Files validated: 30 / 30
Overall accuracy: 100.00%
Total matches: 30 / 30

Perfect domains (1):
  - vendor



""


## Optional: all-JSON validation

Use this only if you want to validate every available field directly from `Json Data`.

This is not the default recommendation for your current setup because your current validation logic for non-vendor fields already uses `OCRed Text` and regex extraction.


In [10]:
# ============================================================================
# OPTIONAL SOLUTION: VALIDATE ALL AVAILABLE FIELDS FROM JSON GROUND TRUTH
# ============================================================================

JSON_PREDICTION_ALIASES = {
    "vendor": ["vendor", "Seller Name", "seller", "seller_name", "vendor_name"],
    "invoice_number": ["invoice_number", "Invoice Number"],
    "invoice_date": ["invoice_date", "Invoice Date"],
    "due_date": ["due_date", "Due Date"],
    "subtotal": ["subtotal", "Net Worth", "net_worth"],
    "tax": ["tax", "VAT", "vat"],
    "total": ["total", "Gross Worth", "gross_worth"],
    "currency": ["currency", "Currency"],
    "status": ["status", "Status"],
}


def get_prediction_value_by_alias(output_row, field):
    """Get a model prediction value for a normalized JSON field."""
    candidates = JSON_PREDICTION_ALIASES.get(field, [field])

    for col in candidates:
        if col in output_row.index:
            return output_row.get(col), col

    return None, None


def validate_with_json_groundtruth(
    output_df,
    batch1_df,
    domains=None,
    tolerance=STRICT_NUMERIC_TOLERANCE,
    fuzzy_threshold=FUZZY_MATCH_THRESHOLD,
):
    """
    Validate model predictions against Json Data for all selected fields.

    Fields use normalized JSON names, for example:
        vendor, invoice_number, invoice_date, subtotal, tax, total
    """
    gt_json_df = build_json_ground_truth_df(batch1_df)
    gt_index = build_gt_index(batch1_df)

    if domains is None:
        candidate_domains = [
            "vendor",
            "invoice_number",
            "invoice_date",
            "subtotal",
            "tax",
            "total",
        ]

        domains = [
            domain for domain in candidate_domains
            if any(col in output_df.columns for col in JSON_PREDICTION_ALIASES.get(domain, [domain]))
        ]

    elif isinstance(domains, str):
        domains = [domains]

    results = initialize_results(domains=domains, total_files=len(output_df))

    for _, output_row in output_df.iterrows():
        filename = filename_only(output_row["filename"])

        if filename not in gt_index:
            results["files_missing"].append(filename)
            continue

        results["files_matched"] += 1
        gt_idx = gt_index[filename]
        gt_json_row = gt_json_df.iloc[gt_idx]

        for domain in domains:
            gt_value = gt_json_row.get(domain)
            pred_value, pred_col = get_prediction_value_by_alias(output_row, domain)
            dr = results["domain_results"][domain]

            if is_missing(gt_value):
                dr["missing_in_ground_truth"] += 1
                dr["details"].append({
                    "filename": filename,
                    "status": "missing_in_gt",
                    "ground_truth": gt_value,
                    "prediction": pred_value,
                    "prediction_column": pred_col,
                })
                continue

            if is_missing(pred_value):
                dr["missing_in_output"] += 1
                dr["mismatches"] += 1
                dr["details"].append({
                    "filename": filename,
                    "status": "missing_in_output",
                    "ground_truth": gt_value,
                    "prediction": pred_value,
                    "prediction_column": pred_col,
                })
                continue

            detail = compare_values(
                domain=domain,
                gt_value=gt_value,
                pred_value=pred_value,
                tolerance=tolerance,
                fuzzy_threshold=fuzzy_threshold,
            )
            detail["filename"] = filename
            detail["prediction_column"] = pred_col

            dr["details"].append(detail)

            if detail["status"] == "match":
                dr["matches"] += 1
            else:
                dr["mismatches"] += 1

    return finalize_results(results)


# Example usage:
json_results = validate_with_json_groundtruth(output_df, batch1_df)
print_validation_report(json_results)
json_mismatches_df = get_mismatch_dataframe(json_results)
display(json_mismatches_df.head(30))



OCR VALIDATION REPORT

Files validated: 30 / 30

--------------------------------------------------------------------------------
OVERALL ACCURACY
--------------------------------------------------------------------------------
Total comparisons: 172
Total matches:      156
Overall accuracy:   90.70%

--------------------------------------------------------------------------------
DOMAIN-BY-DOMAIN ACCURACY
--------------------------------------------------------------------------------


,Domain,Matches,Mismatches,Missing GT,Missing Pred,Accuracy %
0,vendor,30,0,0,0,100.00%
1,invoice_number,30,0,0,0,100.00%
2,invoice_date,30,0,0,0,100.00%
3,subtotal,21,5,4,0,80.77%
4,tax,24,6,0,0,80.00%
5,total,21,5,4,0,80.77%



--------------------------------------------------------------------------------
DETAILED RESULTS
--------------------------------------------------------------------------------

### vendor ###
  ✓ All matches!

### invoice_number ###
  ✓ All matches!

### invoice_date ###
  ✓ All matches!

### subtotal ###
  Showing 5 of 5 issue(s):
    batch1-0053.jpg: mismatch
      GT:         6705.0
      Prediction: 67.05
      Difference: 6637.95
      Prediction column used: subtotal
    batch1-0217.jpg: mismatch
      GT:         6877.19
      Prediction: 7641.32
      Difference: 764.1300000000001
      Prediction column used: subtotal
    batch1-0017.jpg: mismatch
      GT:         361.69
      Prediction: 401.88
      Difference: 40.19
      Prediction column used: subtotal
    batch1-0016.jpg: mismatch
      GT:         28736.0
      Prediction: 287.36
      Difference: 28448.64
      Prediction column used: subtotal
    batch1-0288.jpg: mismatch
      GT:         99.0
      Prediction: 

,Domain,Filename,Ground Truth,Prediction,Status,Difference,Prediction Column
0,subtotal,batch1-0053.jpg,6705.00,67.05,mismatch,6637.95,subtotal
1,subtotal,batch1-0217.jpg,6877.19,7641.32,mismatch,764.13,subtotal
2,subtotal,batch1-0017.jpg,361.69,401.88,mismatch,40.19,subtotal
3,subtotal,batch1-0016.jpg,28736.00,287.36,mismatch,28448.64,subtotal
4,subtotal,batch1-0288.jpg,99.00,0.99,mismatch,98.01,subtotal
5,tax,batch1-0380.jpg,0.00,1704.50,mismatch,1704.50,tax
6,tax,batch1-0053.jpg,671.00,6.71,mismatch,664.29,tax
7,tax,batch1-0016.jpg,2874.00,28.74,mismatch,2845.26,tax
8,tax,batch1-0288.jpg,10.00,0.10,mismatch,9.90,tax
9,tax,batch1-0367.jpg,0.00,8494.48,mismatch,8494.48,tax
